# MATHFI model pipeline with quantitative biomarkers and visualization

In [1]:
# Add project root (the folder that contains "src/") to sys.path
import sys
import time
from pathlib import Path

def add_project_root(marker_dir="src", max_hops=5):
    p = Path.cwd().resolve()
    for _ in range(max_hops):
        if (p / marker_dir).exists():
            sys.path.insert(0, str(p))
            print(f"[OK] Added to sys.path: {p}")
            return
        p = p.parent
    raise RuntimeError(f"Could not find '{marker_dir}' within {max_hops} parents from {Path.cwd()}")

add_project_root()  

[OK] Added to sys.path: D:\Data\Projects\Thesis\Adaptive-Hierarchical-Feature-Modulation-U-Net-Model-for-Retinal-Vessel-Segmentation


In [ ]:
# --- Imports ---
%matplotlib inline
import os, time, json, contextlib
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# --- Your libs ---
from src.data.preprocessing import _iso_resize_and_pad
from apps.streamlit.lib.preprocess import preprocess_image_retina_from_pil  # <--- or your path where preprocess.py lives

# Vessel metrics
from src.retina_biomarkers import (
    to_bool_mask, skeletonize_mask, distance_transform, build_skeleton_graph,
    sample_width_along_skeleton, sample_widths_orthogonal,
    area_density, length_density, caliber_stats, tortuosity_stats, fractal_dimension_boxcount,
    junction_metrics, branching_and_bifurcation_angles, branching_angles_roi, gap_metrics, metrics_by_rings
)

# OD segmentation helpers (from the od_seg package we made)
from src.retina_biomarkers.od_seg import (
    load_refuge_segformer, infer_label_map, extract_disc_mask_safe,
    center_and_pd_with_bounds, show_disc_overlay
)


In [ ]:
# --- Model helpers (use your DPCNConcatUNet and load from .pth) ---
import torch, contextlib

from src.models.wrappers.dpcn_concat_unet import DPCNConcatUNet

def build_dpcn_model(device: str | None = None) -> tuple[torch.nn.Module, str]:
    dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = DPCNConcatUNet(
        in_ch=1,                 # grayscale
        enh_channels=64,         # DPCN internal channels (C)
        iters=6,                 # DPCN iterations (T)
        threshold_mode="scaled_vat",
        half_life=2.0,           # E memory ~2 iterations
        reduce_to=64,            # compress T*C to control memory
        base_kwargs={"cbam_reduction": 16},
        refine_edge=True
    ).to(dev).eval()
    return model, dev

def _strip_prefix_if_present(state_dict: dict, prefix: str = "module.") -> dict:
    if not any(k.startswith(prefix) for k in state_dict.keys()):
        return state_dict
    return {k[len(prefix):] if k.startswith(prefix) else k: v for k, v in state_dict.items()}

def load_weights_into_model(model: torch.nn.Module, ckpt_path: str, device: str) -> None:
    state = torch.load(ckpt_path, map_location=device)
    # handle common wrappers
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    # try strict, then try removing "module." prefix if needed
    try:
        model.load_state_dict(state, strict=True)
        return
    except Exception:
        pass
    try:
        state2 = _strip_prefix_if_present(state, "module.")
        model.load_state_dict(state2, strict=True)
        return
    except Exception as e:
        print("[warn] strict load failed; trying non-strict:", e)
        model.load_state_dict(state2, strict=False)

def load_dpcn_from_ckpt(ckpt_path: str, device: str | None = None) -> tuple[torch.nn.Module, str]:
    model, dev = build_dpcn_model(device)
    load_weights_into_model(model, ckpt_path, dev)
    model.eval()
    return model, dev
